# Day 5 — SIP continuity analysis (at-risk investors)

**Rules:**
- Consider investors with **>= 6 SIP transactions**
- Compute **avg gap (days)** between consecutive SIP `transaction_date`s
- Flag as **at-risk** if `avg_gap_days > 35`


In [1]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px

# --- Repo-root detection ---
_HERE = Path(__file__).resolve() if '__file__' in globals() else Path.cwd()

def _find_repo_root(start: Path) -> Path:
    cand = start
    for _ in range(12):
        if (cand / 'Data' / 'processed' / 'investor_transactions_clean.csv').exists():
            return cand
        if cand.name == 'notebooks':
            parent = cand.parent
            if (parent / 'Data' / 'processed' / 'investor_transactions_clean.csv').exists():
                return parent
        cand = cand.parent
    return start.parent

_REPO_ROOT = _find_repo_root(_HERE)
DATA_DIR = _REPO_ROOT / 'Data' / 'processed'

tx_path = DATA_DIR / 'investor_transactions_clean.csv'
out_path = DATA_DIR / 'sip_continuity_at_risk.csv'

if not tx_path.exists():
    raise FileNotFoundError(f'Missing required file: {tx_path.resolve()}')

df = pd.read_csv(tx_path)
df.head()


,investor_id,transaction_date,amfi_code,transaction_type,amount_inr,state,city,city_tier,age_group,gender,annual_income_lakh,payment_mode,kyc_status
0,INV000001,2024-11-04,120505,SIP,44856,Haryana,Gurugram,T30,36-45,Male,19.9,UPI,Verified
1,INV000001,2025-01-14,148569,Lumpsum,189483,Haryana,Gurugram,T30,36-45,Male,19.9,UPI,Verified
2,INV000001,2025-01-19,125497,SIP,3090,Haryana,Gurugram,T30,36-45,Male,19.9,Cheque,Pending
3,INV000002,2024-03-29,149322,SIP,2830,Maharashtra,Pune,T30,46-55,Male,24.0,Mandate,Verified
4,INV000002,2024-07-14,149323,Lumpsum,153187,Maharashtra,Pune,T30,46-55,Male,24.0,UPI,Verified


In [2]:
required_cols = {'investor_id', 'transaction_date', 'transaction_type'}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError('Missing columns in investor_transactions_clean.csv: ' + str(sorted(missing)))

df['transaction_date'] = pd.to_datetime(df['transaction_date'], errors='coerce')
df = df.dropna(subset=['transaction_date']).copy()

df['transaction_type'] = df['transaction_type'].astype(str).str.upper().str.strip()
sip = df[df['transaction_type'].isin(['SIP'])].copy()
sip.shape


(19716, 13)

In [3]:
MIN_SIP_TX = 6
RISK_THRESHOLD_DAYS = 35

sip = sip.sort_values(['investor_id', 'transaction_date']).copy()

rows = []
for inv_id, g in sip.groupby('investor_id', sort=False):
    dates = g['transaction_date'].sort_values().to_list()
    if len(dates) < MIN_SIP_TX:
        continue

    gaps = [(dates[i + 1] - dates[i]).days for i in range(len(dates) - 1)]
    avg_gap_days = float(np.mean(gaps)) if gaps else np.nan
    at_risk = 'at-risk' if (not np.isnan(avg_gap_days) and avg_gap_days > RISK_THRESHOLD_DAYS) else 'ok'

    rows.append({
        'investor_id': inv_id,
        'sip_tx_count': len(dates),
        'avg_gap_days': avg_gap_days,
        'at_risk': at_risk,
        'max_gap_days': int(np.max(gaps)) if gaps else np.nan,
    })

out_df = pd.DataFrame(rows)
out_df = out_df.sort_values(['at_risk', 'avg_gap_days'], ascending=[False, False])
out_df.head(10)


,investor_id,sip_tx_count,avg_gap_days,at_risk,max_gap_days
167,INV000566,7,35.000000,ok,119
1052,INV003772,9,34.500000,ok,72
214,INV000707,9,34.375000,ok,106
927,INV003382,6,34.200000,ok,85
1172,INV004246,7,34.166667,ok,64
1345,INV004927,7,34.166667,ok,63
451,INV001672,9,34.000000,ok,130
1252,INV004542,7,34.000000,ok,59
1004,INV003650,11,33.900000,ok,86
676,INV002509,10,33.666667,ok,74


In [4]:
out_df.to_csv(out_path, index=False)
print('Wrote:', out_path)

summary = (
    out_df['at_risk'].value_counts(dropna=False)
    .rename_axis('at_risk')
    .reset_index(name='count')
)
summary


Wrote: c:\Mutual Fund Analytics\Data\processed\sip_continuity_at_risk.csv


,at_risk,count
0,at-risk,1332
1,ok,30


In [5]:
at_risk_df = out_df[out_df['at_risk'] == 'at-risk'].copy()
top_at_risk = at_risk_df.sort_values('avg_gap_days', ascending=False).head(25)
top_at_risk


,investor_id,sip_tx_count,avg_gap_days,at_risk,max_gap_days
506,INV001890,6,102.6,at-risk,207
323,INV001156,6,102.4,at-risk,175
1188,INV004296,6,102.2,at-risk,304
910,INV003325,6,101.0,at-risk,155
150,INV000522,6,100.8,at-risk,309
183,INV000608,6,100.2,at-risk,324
503,INV001883,6,99.2,at-risk,203
576,INV002166,6,99.2,at-risk,248
380,INV001367,6,99.0,at-risk,222
406,INV001491,6,98.8,at-risk,260


In [6]:
fig1 = px.histogram(
    out_df,
    x='avg_gap_days',
    nbins=60,
    color='at_risk',
    barmode='overlay',
    opacity=0.65,
    title='Distribution of Avg Gap Days Between SIP Dates (>=6 SIP tx)'
)
fig1.add_vline(
    x=RISK_THRESHOLD_DAYS,
    line_width=2,
    line_dash='dash',
    line_color='red',
    annotation_text=f'Risk threshold = {RISK_THRESHOLD_DAYS} days',
    annotation_position='top left'
)
fig1.show()


In [7]:
fig2 = px.box(
    out_df,
    x='at_risk',
    y='avg_gap_days',
    color='at_risk',
    points=False,
    title=f'Avg Gap Days by At-Risk Group (threshold: {RISK_THRESHOLD_DAYS} days)'
)
fig2.show()
